# _seed_data - pull the public data bundle into OneLake (cloud-to-cloud)

In [ ]:
# _seed_data - cloud-to-cloud: pull the public OneGrid data bundle straight into OneLake.
# The user's laptop never downloads the parquet; this runs on the Fabric Spark driver,
# which has fast egress to OneLake. deploy.ps1 runs this only when no local data/ bundle
# is present (the lightweight wizard case).

WS  = "163ba38c-3869-406f-adb7-37cbc981390c"   # rebound to target workspace id
LH  = "7e08480c-cf8d-4206-901d-38b74dbe35d9"   # rebound to target lakehouse id
BUNDLE_URL = "__DATA_BUNDLE_URL__"             # rebound to the public data bundle (zip) url

import os, sys, time, json, shutil, zipfile, urllib.request

try:
    import notebookutils; fs = notebookutils.fs
except Exception:
    from notebookutils import mssparkutils; fs = mssparkutils.fs

BASE = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{LH}"
DEST = f"{BASE}/Files/solution_import"

tmp     = "/tmp/onegrid_seed"
zippath = "/tmp/onegrid-data.zip"
if os.path.isdir(tmp): shutil.rmtree(tmp)
os.makedirs(tmp, exist_ok=True)

print(f"[seed] data bundle : {BUNDLE_URL}", flush=True)
print(f"[seed] destination : {DEST}", flush=True)
print(f"[seed] downloading ...", flush=True)

t0 = time.time()
last = [0.0]
def _hook(blocks, bs, total):
    now = time.time()
    if now - last[0] >= 2 or (total > 0 and blocks*bs >= total):
        got = blocks*bs/1024/1024
        tot = (total/1024/1024) if total > 0 else 0
        pct = f"{(got/tot*100):5.1f}%" if tot else "  ?  "
        print(f"[seed]   {pct}  {got:8.1f} / {tot:8.1f} MB  ({got/max(now-t0,0.1):5.1f} MB/s)", flush=True)
        last[0] = now
urllib.request.urlretrieve(BUNDLE_URL, zippath, _hook)

mb = os.path.getsize(zippath)/1024/1024
print(f"[seed] downloaded {mb:.1f} MB in {time.time()-t0:.0f}s - extracting...", flush=True)
with zipfile.ZipFile(zippath) as z:
    for _info in z.infolist():
        _name = _info.filename.replace(chr(92), '/')
        if _name.endswith('/'):
            continue
        _dest = os.path.join(tmp, *_name.split('/'))
        os.makedirs(os.path.dirname(_dest), exist_ok=True)
        with z.open(_info) as _src, open(_dest, 'wb') as _out:
            shutil.copyfileobj(_src, _out)

# locate the archive root that holds lakehouse/ and eventhouse/
root = tmp
if not os.path.isdir(os.path.join(root, "lakehouse")):
    cand = os.path.join(tmp, "data")
    if os.path.isdir(os.path.join(cand, "lakehouse")):
        root = cand
print(f"[seed] extracted; archive root = {root}", flush=True)

files_index = {"lakehouse": [], "eventhouse": {}}

for group in ("lakehouse", "eventhouse"):
    src = os.path.join(root, group)
    if not os.path.isdir(src):
        print(f"[seed] WARN group '{group}' not found in bundle - skipping", flush=True)
        continue
    dst = f"{DEST}/{group}"
    # count for a friendly log
    n = sum(len(fnames) for _, _, fnames in os.walk(src))
    print(f"[seed] uploading '{group}' ({n} file(s)) -> {dst}", flush=True)
    try:
        fs.rm(dst, True)   # idempotent: clear any prior partial seed
    except Exception:
        pass
    t1 = time.time()
    fs.cp(f"file:{src}", dst, True)   # recursive local -> OneLake copy (cloud egress)
    print(f"[seed]   '{group}' uploaded in {time.time()-t1:.0f}s", flush=True)
    # build the file index (deploy.ps1 uses the eventhouse part list to drive KQL .ingest)
    for dirpath, _, fnames in os.walk(src):
        for fn in fnames:
            rel = os.path.relpath(os.path.join(dirpath, fn), src).replace("\\", "/")
            if group == "lakehouse":
                files_index["lakehouse"].append(rel)
            elif fn.endswith(".parquet"):
                parts = rel.split("/", 1)
                tbl = parts[0]
                leaf = parts[1] if len(parts) > 1 else rel
                files_index["eventhouse"].setdefault(tbl, []).append(leaf)

idx_local = "/tmp/_files.json"
with open(idx_local, "w") as f:
    json.dump(files_index, f)
fs.cp(f"file:{idx_local}", f"{DEST}/_files.json", False)

eh = {k: len(v) for k, v in files_index["eventhouse"].items()}
print(f"[seed] wrote {DEST}/_files.json", flush=True)
print(f"[seed] lakehouse files: {len(files_index['lakehouse'])} | eventhouse: {eh}", flush=True)
print("[seed] DONE", flush=True)